In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import csv
from pathlib import Path

# Загружаем модель и изображения
model = YOLO('D:/Data_env/Inf_cell_v26L-seg.pt') # модель
input_dir = Path('D:/Data_env/cells')  # папка с изображениями
output_csv = Path('D:/Data_env/cells/results.csv')  # файл, куда запишется результат

# Расширения файлов изображений
image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.tif')

# Создаем таблицу результатов
with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['File Name',
                    'Total Pixels',
                    'Mask Pixels',
                    'Coverage Percentage',
                    'Objects Detected'])

# Поиск всех изображений для анализа
image_files = []
for ext in image_extensions:
    image_files.extend(input_dir.glob(ext))
print(f'Найдено изображений для обработки: {len(image_files)}')

# Основной цикл обработки
for img_path in image_files:
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            print(f'Ошибка загрузки файла (пропущен): {img_path.name}')
            continue

        h, w, _ = img.shape
        total_pixels = h * w

        # Запуск сегментации для конкретного кадра
        results = model(img_path, verbose=False)

        # Создание пустой маски
        all_masks = np.zeros((h, w), dtype=np.uint8)
        object_count = 0

        for result in results:
            if result.masks is not None:
                object_count += len(result.masks.xy)
                # Отрисовка полигонов на маске
                for segments in result.masks.xy:
                    points = np.array(segments, dtype=np.int32)
                    cv2.fillPoly(all_masks, [points], 255)

        # Учет количества закрашенных пикселей маски
        mask_pixels = int(np.sum(all_masks == 255))

        # Вычисление покрытия
        coverage_pct = ((mask_pixels / total_pixels) * 100 if total_pixels > 0 else 0)

        # Заполнение результатов текущей обработки изображения в CSV
        with open(output_csv, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([img_path.name,
                            total_pixels,
                            mask_pixels,
                            round(coverage_pct, 4),
                            object_count])

        print(f'✅ Обработан {img_path.name} | Покрытие: {coverage_pct:.2f}%')

    except Exception as e:
        print(f'Ошибка при обработке {img_path.name}: {e}')

# Выводим результат
print(f'Обработка завершена. Результаты сохранены в {output_csv.resolve()}')

Найдено изображений для обработки: 14
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_10.jpg | Покрытие: 53.94%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_4.jpg | Покрытие: 51.07%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_5.jpg | Покрытие: 64.34%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_6.jpg | Покрытие: 59.96%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_7.jpg | Покрытие: 69.61%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_8.jpg | Покрытие: 37.75%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_9.jpg | Покрытие: 62.14%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_10.jpg | Покрытие: 80.43%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_4.jpg | Покрытие: 70.02%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_5.jpg | Покрытие: 66.18%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_6.jpg | Покрытие: 63.84%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_7.jpg | Покрытие: 63.00%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_8.jpg | Покрытие: 79.59%
✅ Обработан Glycyrrhiza+

---